In [3]:
from sklearn.datasets import make_regression
import pandas as pd 

In [4]:
x,y = make_regression( n_samples=2000, random_state=42, noise=20, n_features=6)
df = pd.DataFrame(x, columns=[ f'feature_{i}' for i in range(6)])
df["target"] = y 
df.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,target
0,-0.573703,-1.063259,-0.459605,-2.729404,0.005205,-0.553241,-195.165598
1,-0.054295,-0.272724,-0.245743,-0.259591,-1.503143,-2.696887,-368.807933
2,1.718364,0.268812,-0.323168,1.121689,0.229095,-0.518638,94.042984
3,-0.310308,0.379768,1.237654,-0.970124,-0.397558,-0.968046,-85.316289
4,-0.039162,0.192652,-2.378432,-1.438720,-0.225263,0.365394,-156.642318


In [5]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

In [6]:
(train_df.shape,valid_df.shape,test_df.shape)

((1400, 7), (300, 7), (300, 7))

In [7]:
from sklearn.linear_model import LinearRegression

x_train = train_df.drop(columns=["target"], axis = 1)
y_train = train_df["target"]

x_val = valid_df.drop(columns=["target"], axis = 1)
y_val = valid_df["target"]

In [8]:
model = LinearRegression()

In [9]:
model.fit(x_train,y_train)

val_pred = model.predict(x_val)

In [10]:
from sklearn.metrics import mean_squared_error , mean_absolute_error, r2_score
import numpy as np 
mse = mean_squared_error(y_val, val_pred)
mae = mean_absolute_error(y_val,val_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_val, val_pred)

print(mse, mae, rmse, r2)

451.95509981538333 16.864623219902082 21.259235635727435 0.9782098559239438


In [17]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from scipy import stats

residuels = y_val - val_pred

# Create histogram
fig = px.histogram(residuels, nbins=50, title="Residuals Distribution Plot")
fig.update_traces(marker=dict(line=dict(color="white", width=1)))

# Calculate normal distribution curve
mu = residuels.mean()
sigma = residuels.std()
x_range = np.linspace(residuels.min(), residuels.max(), 100)
normal_curve = stats.norm.pdf(x_range, mu, sigma)

# Scale the curve to match histogram height
# We need to multiply by bin width and total count
bin_width = (residuels.max() - residuels.min()) / 50
normal_curve_scaled = normal_curve * len(residuels) * bin_width

# Add the normal distribution curve
fig.add_trace(go.Scatter(
    x=x_range, 
    y=normal_curve_scaled,
    mode='lines',
    name='Normal Distribution',
    line=dict(color='black', width=3)
))

fig.update_layout(showlegend=True)
fig.show()

In [14]:
import plotly.graph_objects as go
import plotly.express as px
fig1 = px.scatter(x=val_pred, y=residuels, 
                  color=residuels,  # Color by residual value
                  title="Residuals vs Predicted",
                  labels={'x': 'Predicted', 'y': 'Residual'})  # Red-Blue scale
fig1.add_hline(y=0, line_dash="dash", line_color="black")

fig1.show()
